In [1]:
from curl_cffi import requests as curl_requests
import pandas as pd
import json
import time
import random
import os

In [7]:
# ============ CHANGE THESE AS NEEDED ============

CITY = "gurgaon"
EXPECTED_TYPE = "Builder Floor"  # Exact value from 99acres JSON

MAX_PAGES_PER_LOCALITY = 70

# Throttling
MIN_DELAY = 1
MAX_DELAY = 3
BATCH_PAUSE = 20
BATCH_SIZE = 10

# ============ AUTO-CONFIGURED ============

# NOTE: The builder floor URL on 99acres does NOT filter server-side.
# It returns ALL property types. Our code filter (EXPECTED_TYPE) handles
# the actual filtering. Locality counts will appear inflated — that's normal.
URL_PREFIX = "independent-builder-floors-in"
CITY_BASE_URL = f"https://www.99acres.com/{URL_PREFIX}-{CITY}-ffid"
OUTPUT_DIR = "../../data/web_scraping"
OUTPUT_FILE = os.path.join(OUTPUT_DIR, f"builder_floor_{CITY}.csv")
PROGRESS_FILE = os.path.join(OUTPUT_DIR, f".progress_builder_floor_{CITY}.json")

os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"Property Type: Builder Floor")
print(f"Filter: PROPERTY_TYPE == '{EXPECTED_TYPE}'")
print(f"City: {CITY}")
print(f"City URL: {CITY_BASE_URL}")
print(f"Output: {OUTPUT_FILE}")
print(f"\n⚠️  Note: 99acres' builder floor URL doesn't filter server-side.")
print(f"Locality counts will be inflated but extraction is correct.")

Property Type: Builder Floor
Filter: PROPERTY_TYPE == 'Builder Floor'
City: gurgaon
City URL: https://www.99acres.com/independent-builder-floors-in-gurgaon-ffid
Output: ../../data/web_scraping\builder_floor_gurgaon.csv

⚠️  Note: 99acres' builder floor URL doesn't filter server-side.
Locality counts will be inflated but extraction is correct.


In [8]:
def load_scraped_ids(output_file):
    if os.path.exists(output_file):
        df = pd.read_csv(output_file)
        ids = set(df["property_id"].dropna().astype(str).tolist())
        print(f"Resuming: {len(ids)} properties already scraped.")
        return ids
    print("Fresh start.")
    return set()

def save_page_data(records, output_file):
    if not records:
        return
    df = pd.DataFrame(records)
    write_header = not os.path.exists(output_file)
    df.to_csv(output_file, mode="a", header=write_header, index=False)

def load_progress():
    if os.path.exists(PROGRESS_FILE):
        with open(PROGRESS_FILE, "r") as f:
            progress = json.load(f)
        return set(progress.get("completed_localities", []))
    return set()

def save_progress(completed_localities):
    with open(PROGRESS_FILE, "w") as f:
        json.dump({"completed_localities": sorted(completed_localities)}, f)

scraped_ids = load_scraped_ids(OUTPUT_FILE)
completed_localities = load_progress()
print(f"Completed localities: {len(completed_localities)}")

Resuming: 9210 properties already scraped.
Completed localities: 100


In [9]:
FACING_MAP = {
    "1": "East", "2": "West", "3": "North", "4": "South",
    "5": "North-East", "6": "North-West", "7": "South-East", "8": "South-West",
}

FURNISH_MAP = {
    "1": "Furnished", "2": "Semi-Furnished", "3": "Unfurnished", "4": "Unfurnished",
}

OVERLOOKING_MAP = {
    "1": "Park/Garden", "2": "Main Road", "3": "Club",
    "4": "Pool", "5": "Others", "7": "Sea facing",
}

def map_codes(code_str, mapping):
    if not code_str:
        return ""
    codes = str(code_str).split(",")
    names = [mapping.get(c.strip(), c.strip()) for c in codes]
    return ", ".join(names)

def extract_property(p):
    landmarks = p.get("LANDMARK_DETAILS") or []
    nearby = [lm.get("name", "") for lm in landmarks if lm.get("name")]

    area_val = p.get("AREA", "")
    area_type = (p.get("AREA_TYPE") or "").replace("_", " ").title()
    area_with_type = f"{area_val} {area_type}".strip() if area_val else ""

    floor_num = p.get("FLOOR_NUM", "")
    total_floor = p.get("TOTAL_FLOOR", "")
    floor_info = f"{floor_num} of {total_floor} Floors" if floor_num and total_floor else str(floor_num)

    facing_code = str(p.get("FACING", ""))
    facing = FACING_MAP.get(facing_code, facing_code)

    furnish_code = str(p.get("FURNISH", ""))
    furnish = FURNISH_MAP.get(furnish_code, furnish_code)

    overlooking = map_codes(p.get("OVERLOOKING", ""), OVERLOOKING_MAP)
    features = p.get("USP_V2_FH_REMOVED", "")

    map_details = p.get("MAP_DETAILS") or {}
    latitude = map_details.get("LATITUDE", "")
    longitude = map_details.get("LONGITUDE", "")

    parking = p.get("RESERVED_PARKING", "")

    return {
        "property_name": p.get("PROP_HEADING", ""),
        "link": "https://www.99acres.com" + p.get("PD_URL", ""),
        "society": p.get("SOCIETY_NAME") or p.get("BUILDING_NAME", ""),
        "price": p.get("FORMATTED_PRICE") or p.get("PRICE", ""),
        "area": p.get("LOCALIZED_PRICE_SQFT_TEXT", ""),
        "areaWithType": area_with_type,
        "carpetArea": p.get("CARPET_AREA", ""),
        "bedRoom": p.get("BEDROOM_NUM", ""),
        "bathroom": p.get("BATHROOM_NUM", ""),
        "balcony": p.get("BALCONY_NUM", ""),
        "address": p.get("LOCALITY", ""),
        "floorNum": floor_info,
        "facing": facing,
        "overlooking": overlooking,
        "agePossession": p.get("AGE", ""),
        "cornerProperty": p.get("CORNER_PROPERTY", ""),
        "furnishing": furnish,
        "parking": parking,
        "nearbyLocations": nearby if nearby else "",
        "description": p.get("DESCRIPTION", ""),
        "features": features,
        "latitude": latitude,
        "longitude": longitude,
        "property_id": p.get("PROP_ID", ""),
    }

print("Extractor ready.")

Extractor ready.


In [10]:
def discover_localities(city_url):
    print(f"Discovering localities from: {city_url}")
    resp = curl_requests.get(city_url, impersonate="chrome120", timeout=30)

    if resp.status_code != 200:
        raise Exception(f"Failed to fetch city page: HTTP {resp.status_code}")

    html = resp.text
    marker = "window.__initialData__="
    idx = html.find(marker)
    if idx == -1:
        raise Exception("No __initialData__ found on city page")

    decoder = json.JSONDecoder()
    data, _ = decoder.raw_decode(html[idx + len(marker):])
    localities = data["srp"]["pageData"]["facets"]["LOCALITY_ID"]

    results = []
    for loc in localities:
        slug = (loc["label"].lower()
                .strip()
                .replace(" ", "-")
                .replace("--", "-")
                .strip("-"))
        if slug.endswith(f"-{CITY}"):
            slug = slug[:-len(f"-{CITY}")]
        results.append({
            "id": loc["id"],
            "label": loc["label"],
            "count": loc["count"],
            "slug": slug,
        })

    results.sort(key=lambda x: x["count"], reverse=True)
    return results

localities = discover_localities(CITY_BASE_URL)
total_listed = sum(loc["count"] for loc in localities)
pending = [loc for loc in localities if loc["slug"] not in completed_localities]

print(f"\nFound {len(localities)} localities (~{total_listed} total listings, counts inflated)")
print(f"Already completed: {len(completed_localities)}")
print(f"Remaining: {len(pending)}")

Discovering localities from: https://www.99acres.com/independent-builder-floors-in-gurgaon-ffid


Exception: Failed to fetch city page: HTTP 417

In [ ]:
MAX_CONSECUTIVE_ERRORS = 5
total_new = 0
rate_limited = False
total_requests = 0

pending = [loc for loc in localities if loc["slug"] not in completed_localities]

for loc_idx, locality in enumerate(pending):
    slug = locality["slug"]
    label = locality["label"]
    base_url = f"https://www.99acres.com/{URL_PREFIX}-{slug}-{CITY}-ffid"

    print(f"\n{'='*60}")
    print(f"LOCALITY {loc_idx+1}/{len(pending)}: {label}")
    print(f"URL: {base_url}")
    print(f"{'='*60}")

    locality_new = 0
    consecutive_errors = 0

    for page in range(1, MAX_PAGES_PER_LOCALITY + 1):
        url = f"{base_url}-page-{page}" if page > 1 else base_url

        try:
            resp = curl_requests.get(url, impersonate="chrome120", timeout=30)
            total_requests += 1

            if resp.status_code != 200:
                if resp.status_code in (410, 404):
                    if page == 1:
                        print(f"  Page {page}: HTTP {resp.status_code} — skipping locality")
                    else:
                        print(f"  Page {page}: HTTP {resp.status_code} — end of pages")
                    break
                consecutive_errors += 1
                print(f"  Page {page}: HTTP {resp.status_code} (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    print(f"  ⛔ Rate limited! Stopping.")
                    rate_limited = True
                    break
                continue

            html = resp.text
            marker = "window.__initialData__="
            idx = html.find(marker)

            if idx == -1:
                consecutive_errors += 1
                print(f"  Page {page}: No JSON (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    rate_limited = True
                    break
                continue

            decoder = json.JSONDecoder()
            data, _ = decoder.raw_decode(html[idx + len(marker):])

            if "srp" not in data:
                consecutive_errors += 1
                print(f"  Page {page}: No 'srp' key (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
                if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                    rate_limited = True
                    break
                continue

            properties = data["srp"]["pageData"]["properties"]
            individual = [
                p for p in properties
                if p.get("entityType") is None and p.get("PROPERTY_TYPE") == EXPECTED_TYPE
            ]

            if not individual and page > 1:
                print(f"  Page {page}: 0 builder floors — end of pages")
                break

            new_records = []
            for p in individual:
                pid = str(p.get("PROP_ID", ""))
                if pid and pid not in scraped_ids:
                    new_records.append(extract_property(p))
                    scraped_ids.add(pid)

            save_page_data(new_records, OUTPUT_FILE)
            locality_new += len(new_records)
            total_new += len(new_records)
            consecutive_errors = 0

            print(f"  Page {page}: {len(individual)} builder floors, {len(new_records)} new — locality: {locality_new}, total: {total_new}")

        except Exception as e:
            consecutive_errors += 1
            print(f"  Page {page}: Error — {e} (error {consecutive_errors}/{MAX_CONSECUTIVE_ERRORS})")
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                rate_limited = True
                break
            continue

        # Throttle
        if total_requests % BATCH_SIZE == 0:
            time.sleep(BATCH_PAUSE)
        else:
            time.sleep(random.uniform(MIN_DELAY, MAX_DELAY))

    if not rate_limited:
        completed_localities.add(slug)
        save_progress(completed_localities)
        print(f"  ✅ {label} done — {locality_new} new builder floors")
    else:
        print(f"\n⛔ Stopped due to rate limiting after {total_new} new properties.")
        print(f"✅ Progress saved ({len(completed_localities)} localities done). Just re-run to resume.")
        break

    if loc_idx < len(pending) - 1:
        pause = random.uniform(5, 10)
        print(f"  Switching locality (pause {pause:.0f}s)...")
        time.sleep(pause)

print(f"\n{'='*60}")
print(f"SUMMARY: {total_new} new builder floors scraped across {len(completed_localities)} localities")
print(f"Total unique properties: {len(scraped_ids)}")
print(f"Output: {OUTPUT_FILE}")
print(f"{'='*60}")


LOCALITY 1/55: Sector 68 Gurgaon
URL: https://www.99acres.com/independent-builder-floors-in-sector-68-gurgaon-ffid
  Page 1: 0 builder floors, 0 new — locality: 0, total: 0
  Page 2: 0 builder floors — end of pages
  ✅ Sector 68 Gurgaon done — 0 new builder floors
  Switching locality (pause 6s)...

LOCALITY 2/55: Malibu Town
URL: https://www.99acres.com/independent-builder-floors-in-malibu-town-gurgaon-ffid
  Page 1: 25 builder floors, 25 new — locality: 25, total: 25
  Page 2: 24 builder floors, 24 new — locality: 49, total: 49
  Page 3: 24 builder floors, 24 new — locality: 73, total: 73
  Page 4: 18 builder floors, 18 new — locality: 91, total: 91
  Page 5: 16 builder floors, 16 new — locality: 107, total: 107
  Page 6: 17 builder floors, 15 new — locality: 122, total: 122
  Page 7: 17 builder floors, 15 new — locality: 137, total: 137
  Page 8: 24 builder floors, 20 new — locality: 157, total: 157
  Page 9: 23 builder floors, 18 new — locality: 175, total: 175
  Page 10: 18 build

In [11]:
df = pd.read_csv(OUTPUT_FILE)
print(f"Total rows: {len(df)}")

before = len(df)
df = df.drop_duplicates(subset="property_id", keep="first")
after = len(df)

if before != after:
    print(f"Removed {before - after} duplicates.")
    df.to_csv(OUTPUT_FILE, index=False)

print(f"Final: {len(df)} unique properties")
print(f"\nColumns ({len(df.columns)}): {list(df.columns)}")
print(f"\nNull counts:")
print(df.isnull().sum())
df.head()

Total rows: 9210
Final: 9210 unique properties

Columns (24): ['property_name', 'link', 'society', 'price', 'area', 'areaWithType', 'carpetArea', 'bedRoom', 'bathroom', 'balcony', 'address', 'floorNum', 'facing', 'overlooking', 'agePossession', 'cornerProperty', 'furnishing', 'parking', 'nearbyLocations', 'description', 'features', 'latitude', 'longitude', 'property_id']

Null counts:
property_name         0
link                  0
society            3363
price                 0
area                 92
areaWithType          0
carpetArea         4840
bedRoom               0
bathroom              0
balcony               1
address               0
floorNum              0
facing                0
overlooking        1051
agePossession         0
cornerProperty     5416
furnishing            0
parking              50
nearbyLocations     131
description           0
features            164
latitude              0
longitude             0
property_id           0
dtype: int64


,property_name,link,society,price,area,areaWithType,carpetArea,bedRoom,bathroom,balcony,...,agePossession,cornerProperty,furnishing,parking,nearbyLocations,description,features,latitude,longitude,property_id
0,"3 BHK Builder Floor in Sector-33 Sohna, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,Central Park Flower Valley Flamingo Floors,2.97 Cr,₹1.55 L /sqyd,1728 sqft Super Area,NaN,3,3,2.0,...,5,NaN,Unfurnished,"{""C"":2}","['Spazedge IT Park', 'Omaxe City Centre', 'JMD...",Independent builder floor premium\nHere is an ...,"['Overlooking Park', 'Overlooking Main Road', ...",28.283817,77.077636,V90137564
1,"3 BHK Builder Floor in Sohna, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,ATS Bonheur Avenue,1.53 Cr,₹1.13 L /sqyd,1214 sqft Super Area,1105.0,3,2,2.0,...,6,Y,Unfurnished,"{""O"":1,""C"":1}","['Airia Mall', 'Badshahpur Sohna Road Highway'...",3 bhk luxury floor at sohna road south gurgaon...,"['Overlooking Park', 'Overlooking Main Road', ...",28.285526,77.064149,H90017582
2,"3 BHK Builder Floor in Sohna, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,Signature Global Daxin Vistas,2.04 Cr,"₹15,975 /sqft",1277 sqft Carpet Area,1277.0,3,3,3.0,...,5,NaN,Semi-Furnished,"{""C"":2}","['National Highway 248A', 'Tau Devilal Stadium...",Welcome to a home that instantly stands out fr...,"['North-East Facing', 'Vaastu Compliant', 'Vis...",28.332976,77.067894,D86965890
3,"3 BHK Builder Floor in Sector-35 Sohna, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,Green Valley Independent Floors,1.68 Cr,₹1.14 L /sqyd,1323 sqft Super Area,NaN,3,3,2.0,...,5,NaN,Unfurnished,"{""C"":1}","['S.R.S. Hospital and Critical Care Unit', 'Th...","Located in green valley independent floors, se...","['Gated Society', 'Overlooking Park', 'North F...",28.419477,76.999628,I90695536
4,"3 BHK Builder Floor in Sohna, Gurgaon",https://www.99acres.com/3-bhk-bedroom-independ...,Signature Global Daxin Vistas,2.35 Cr,₹1.14 L /sqyd,1850 sqft Super Area,1221.0,3,3,2.0,...,5,NaN,Semi-Furnished,"{""C"":1}","['National Highway 248A', 'Tau Devilal Stadium...","This 1st-Floor, low-Rise builder floor in sign...","['Gated Society', 'Overlooking Park', 'Overloo...",28.332976,77.067894,E90903990
